# Zadanie domowe

W przypadku obrazów w odcieniach szarości pojedynczy piksel z zakresu [0; 255] reprezentowany jest jako 8-bitowa liczba bez znaku.
Pewnym rozszerzeniem analizy sposobu reprezentacji obrazu może być następujący eksperyment.
Załóżmy, że z każdego z 8 bitów możemy stworzyć pojedynczy obraz binarny (ang. _bit-plane slicing_).
Dla obrazka _100zloty.jpg_ (https://raw.githubusercontent.com/vision-agh/poc_sw/master/02_Point/100zloty.jpg) stwórz 8 obrazów, z których każdy powinien zawierać jedną płaszczyznę bitową.
Podpowiedź $-$ warto sprawdzić, jak realizuje się bitowe operacje logiczne.
Zastosowanie takiej dekompozycji obrazu pozwala na analizę ,,ważności'' poszczególnych bitów.
Jest to użyteczne w kwantyzacji, ale także w kompresji.

W drugim etapie zadania proszę spróbować odtworzyć obraz oryginalny z mniejszej liczby obrazów binarnych.
Warto zacząć od dwóch najbardziej znaczących bitów, a później dodawać kolejne.
Należy utworzyć co najmniej trzy wersje zrekonstruowanych obrazów.
Podpowiedź $-$ rekonstrukcja obrazu to mnożenie przez odpowiednią potęgę liczby 2 (przesunięcie bitowe) oraz dodawanie.

In [ ]:
import os
import matplotlib.pyplot as plt
import cv2
import numpy as np
if not os.path.exists('100zloty.jpg'):
    !wget https://raw.githubusercontent.com/vision-agh/poc_sw/master/02_Point/100zloty.jpg



In [ ]:
sto = cv2.imread('100zloty.jpg',cv2.IMREAD_GRAYSCALE) # Inaczej dostaniemy obraz RGB, ciezki do przeanalizowania zgodnie z trescia zadania
plt.imshow(sto,cmap='gray')
plt.axis('off')
plt.title('Oryginał')

##############################
def hex_to_bit(num):
    binary = format(num, '08b')
    return [i for i in binary] # lista 8 bitow kazdy jeden poleci do jednej macierzy warstwy

def transform_list(list):
    for i in range(0,len(list)):
        list[i] = "0"*i + list[i] + "0"*(7-i) 
        list[i] = np.uint8(int(list[i], 2)) # Transformuje najpierw binarke na inta a pozniej na uint8 zeby pasowalo do finalnej macierzy
    return None

def bit_slicing(image): # obrzydliwie wolne (kolo 20sekund na moim pc)
    rows,cols = image.shape[:2]
    bit_planes = [np.zeros((rows, cols), dtype=np.uint8) for _ in range(8)] # Bedzie zwracac liste 8 macierzy (obrazow)
    
    for i in range(0,rows):
        for j in range(0,cols):
            bits = hex_to_bit(image[i][j])
            transform_list(bits)
            for k in range(0,8): # range stały bo obraz ma miec uint8
                bit_planes[7-k][i][j] = bits[k] # w ten sposob najbardziej znaczace bity sa w macierzy 7 bit_planes

    return bit_planes
###############################

def better_bit_slicing(image): # duzo lepsze z wykorzystaniem operacji na bitach calej macierzy, (kolo 5sekund na moim pc)
    return [((image >> i) & 1) * 255 for i in range(0,8)]

def reconstruct(layers, indices):
    reconstructed = np.zeros(layers[0].shape, dtype=np.uint8)
    
    for i in indices:
        reconstructed += (layers[i] // 255).astype(np.uint8) * (2**i)
        
    return reconstructed
        
layers = better_bit_slicing(sto)
plt.figure(figsize=(5, 10)) 

for i in range(8):
    plt.subplot(8, 1, i + 1)
    plt.imshow(layers[7-i], cmap='gray')
    plt.axis('off')

sto123 = reconstruct(layers,[0,1,2])
sto456 = reconstruct(layers,[3,4,5])
sto678 = reconstruct(layers,[5,6,7])

plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(sto678, cmap='gray')
plt.title('3 Najbardziej znaczące bity')

plt.subplot(1, 3, 2)
plt.imshow(sto456, cmap='gray')
plt.title('3 Bity o średnim znaczneiu')

plt.subplot(1, 3, 3)
plt.imshow(sto123, cmap='gray')
plt.title('3 Bity o najmniejszym znaczneiu')



In [ ]:
# Na podstawe obserwacji widać już czemu przy kompresji obrazów usuwa się mniej znaczące bity, obraz z 3 najbardziej znaczacych bitów niewiele różni sie od oryginalu